# UrbanPulse - Bangalore Traffic & AQI EDA and Cleaning

This notebook outlines the comprehensive data cleaning, processing, and merging pipeline for the UrbanPulse dashboard project. It combines boundaries, Air Quality Index (AQI), traffic flow snapshots, and weather parameters into a single master analysis dataset.

## Scaffolding & Pipeline Outline
1. **Load Datasets**: Imports historical files for AQI, traffic, and weather.
2. **AQI Cleaning**: Forward fills missing AQI values up to 2 hours and removes outliers (> 500 AQI).
3. **Traffic Feature Engineering**: Extracts temporal features (`hour` and `is_weekend`).
4. **Asof Merge**: Merges all three sources sequentially on the nearest hour while enforcing location-specific alignment.
5. **Lag Feature Creation**: Computes a 1-hour lag on traffic congestion levels (`traffic_aqi_lag1`).
6. **Save Master Dataset**: Exports the clean database to `data/processed/master_df.csv`.
7. **Verification & Heatmap**: Visualizes dataset structure, properties, and checks for remaining missing values.

In [1]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Set aesthetic plotting styles
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print("Libraries successfully imported.")

### Step 1: Load Scraped Historical Datasets

In [2]:
# Define data directory path relative to the notebook location
data_dir = "../data/processed"

aqi = pd.read_csv(os.path.join(data_dir, "aqi_history.csv"))
traffic = pd.read_csv(os.path.join(data_dir, "traffic_history.csv"))
weather = pd.read_csv(os.path.join(data_dir, "weather_history.csv"))

print(f"Loaded AQI dataset: {aqi.shape} (rows, columns)")
print(f"Loaded Traffic dataset: {traffic.shape} (rows, columns)")
print(f"Loaded Weather dataset: {weather.shape} (rows, columns)")

### Step 2: Clean AQI Data
- Parse dates to `datetime` objects and sort chronologically.
- Forward fill (`ffill`) missing values within a **2-hour window** limit to prevent extreme gap leaks, and drop the remaining unresolved missing rows.
- Remove extreme outlier readings (where `AQI > 500`) to guarantee high-quality analysis.

In [3]:
# Convert to datetime and sort
aqi['timestamp'] = pd.to_datetime(aqi['timestamp'])
aqi = aqi.sort_values('timestamp')

print(f"AQI Missing Values before cleaning: {aqi['AQI'].isnull().sum()}")

# Forward fill AQI within a 2-hour limit per station
aqi['AQI'] = aqi.groupby('station')['AQI'].ffill(limit=2)

# Drop any remaining NaNs
aqi = aqi.dropna(subset=['AQI'])
print(f"AQI Rows after fill & drop: {len(aqi)}")

# Remove extreme outliers (> 500 AQI)
outliers = aqi[aqi['AQI'] > 500]
print(f"Removed {len(outliers)} AQI outliers (> 500)")
aqi = aqi[aqi['AQI'] <= 500]

### Step 3: Traffic Feature Engineering
- Parse traffic dates to `datetime` objects and sort chronologically.
- Extract the `hour` of the day (`0-23`).
- Detect if a day falls on the weekend using `is_weekend` (where `1` represents Saturday or Sunday, and `0` represents a weekday).

In [4]:
traffic['timestamp'] = pd.to_datetime(traffic['timestamp'])
traffic = traffic.sort_values('timestamp')

# Feature Engineering: Extract hour and weekend indicator
traffic['hour'] = traffic['timestamp'].dt.hour
traffic['is_weekend'] = (traffic['timestamp'].dt.weekday >= 5).astype(int)

print("Traffic temporal features created successfully.")
traffic[['timestamp', 'junction_name', 'hour', 'is_weekend']].head(3)

### Step 4: Merge Datasets on Nearest Hour Using `pandas.merge_asof`
We need to match the spatial dimensions of traffic junctions with our air quality monitoring locations first. To align them, we register a mapping dictionary:
- `Silk Board` -> `Silk Board`
- `Hebbal` -> `Hebbal`
- `Whitefield` -> `Whitefield`
- `Koramangala` -> `BTM Layout` (adjacent residential center)
- `Marathahalli` -> `Peenya` (mapped for localized evaluation)

Once spatial alignments match, we perform a chronological left join (`merge_asof`) on `timestamp` matching on `location`, then merge the regional `weather` data globally on `timestamp`.

In [5]:
# Spatial alignment map
junction_to_station = {
    "Silk Board": "Silk Board",
    "Hebbal": "Hebbal",
    "Whitefield": "Whitefield",
    "Koramangala": "BTM Layout",
    "Marathahalli": "Peenya"
}

traffic['location'] = traffic['junction_name'].map(junction_to_station)
aqi['location'] = aqi['station']

# Both datasets must be strictly sorted by the join key ('timestamp') before merge_asof
traffic = traffic.sort_values('timestamp')
aqi = aqi.sort_values('timestamp')

# Step A: Merge Traffic and AQI on nearest timestamp matching locations
merged_df = pd.merge_asof(
    traffic,
    aqi,
    on='timestamp',
    by='location',
    direction='nearest'
)

# Step B: Merge with Bangalore regional weather on nearest timestamp globally
weather['timestamp'] = pd.to_datetime(weather['timestamp'])
weather = weather.sort_values('timestamp')

master_df = pd.merge_asof(
    merged_df.sort_values('timestamp'),
    weather,
    on='timestamp',
    direction='nearest'
)

print(f"Unified Master Dataset shape: {master_df.shape}")

### Step 5: Create Feature 'traffic_aqi_lag1' (Congestion Level Lagged 1 Hour)
To model dynamic delays, we compute `traffic_aqi_lag1` which records the traffic congestion level at the same location exactly 1 hour before. This is created by grouping on `location` and shifting chronologically.

In [6]:
# Sort chronologically per location first to guarantee lag alignment
master_df = master_df.sort_values(['location', 'timestamp'])

# Shift congestion level by 1 hour (1 row step) per location
master_df['traffic_aqi_lag1'] = master_df.groupby('location')['congestion_level'].shift(1)

print("Lag feature 'traffic_aqi_lag1' created successfully.")
master_df[['timestamp', 'location', 'congestion_level', 'traffic_aqi_lag1']].head(4)

### Step 6: Save Master Dataset to Disk

In [7]:
master_save_path = os.path.join(data_dir, "master_df.csv")
master_df.to_csv(master_save_path, index=False)
print(f"Successfully saved master dataset to: {master_save_path}")

### Step 7: Verify Results & Heatmap Analysis
Inspect the structure, column properties, counts, and generate a missing values heatmap with seaborn to confirm the quality of our master dataset.

In [8]:
print("=== Master Dataset Info ===")
print(master_df.info())

print("\n=== Master Dataset Head (First 5 rows) ===")
display(master_df.head())

# Generate missing value heatmap
plt.figure(figsize=(12, 6))
sns.heatmap(master_df.isnull(), cbar=False, yticklabels=False, cmap="viridis")
plt.title("UrbanPulse Master Dataset - Missing Value Heatmap", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Columns", fontsize=12)
plt.ylabel("Rows (Aggregated)", fontsize=12)
plt.tight_layout()
plt.show()